In [103]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast
import re
from Bio import PDB
from Bio.PDB import PDBParser
import time
import scipy



In [104]:
'''Step 1: Counting n-neighbours for atoms'''

def convert_cif_to_pdb(cif_file_path, pdb_file_path):
    '''
    Takes in a .cif file, and returns a .pdb file.

    Both file paths should be specified below    
    '''

    parser = PDB.MMCIFParser()
    structure = parser.get_structure('protein', cif_file_path)

    io = PDB.PDBIO()
    io.set_structure(structure)
    io.save(pdb_file_path)

def extract_pdb_info(pdb_file_path):
    """
    Reads a PDB file and extracts 3D spatial coordinates and amino acids.

    Input: pdb_file_path (str): Path to the PDB file.

    Output:  A list of dictionaries, each containing information about each atom.
    """
    # Create a PDBParser object
    parser = PDBParser(PERMISSIVE=1)
    
    # Parse the structure from the PDB file
    structure = parser.get_structure('protein', pdb_file_path)
    
    # List to hold the extracted information
    atom_info_list = []

    # Extract information from the structure
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    atom_info = {
                        'model_id': model.id,
                        'chain_id': chain.id,
                        'residue_name': residue.resname,
                        'residue_id': residue.id[1],
                        'atom_name': atom.name,
                        'atom_coords': atom.coord.tolist()
                    }
                    atom_info_list.append(atom_info)

    return atom_info_list


def count_neighboring_atoms(data):
    '''
    Adds 'neighbour_count': n for each atom data dictionary

    Input: A list of dictionaries, each containing information about each atom.
    
    Output: The same list of dictionaries, appended with 'neighbour_count': n for each atom data dictionary
    '''
    start_time = time.time()

    def distance_np(coord1, coord2):
        return np.linalg.norm(coord1 - coord2)

    def atom_nearest_neighbours(data):

        coords = np.array([atom['atom_coords'] for atom in data])
        neighbour_counts = np.zeros(len(data), dtype=int)
        
        for i in range(len(coords)):
            for j in range(i+1, len(coords)):
                distances = distance_np(coords[i],coords[j])
                if distances <= 3:
                    neighbour_counts[i] += 1

        for i in range(len(neighbour_counts)):
            data[i]['neighbour_count'] = neighbour_counts[i]
        # print(neighbour_counts)
        
        return data

    neighbour_counts = atom_nearest_neighbours(data)

    end_time = time.time()
    elapsed = end_time - start_time
    print(elapsed)

    '''
    #delete irrelevant ids in data if required at this step.
    for entry in range(len(neighbour_counts)):
        del neighbour_counts[entry]['model_id']
        del neighbour_counts[entry]['chain_id']
        # del neighbour_counts[entry]['residue_name']
        del neighbour_counts[entry]['residue_id']
    '''
        
    return neighbour_counts

In [105]:
'''Step 2: Counting n-neighbours for amino acids'''

def count_neighbouring_amino_acids(data):
    '''
    Trim atom dictionary to retain one entry per amino acid with the highest neighbour_count

    Takes in dictionary of atom data appended with key-value pair 'neighbour_count': n  

    Returns dictionary of amino acids with key-value pair 'neighbour_count': n 
    '''

    # Dictionary to store the highest 'neighbour_count' for each 'residue_id'
    max_neighbour_counts = {}

    # Iterate through the data to find the highest 'neighbour_count' for each 'residue_id'
    for atom in data:
        residue_id = atom['residue_id']
        neighbour_count = atom.get('neighbour_count', 0)
        if residue_id not in max_neighbour_counts or neighbour_count > max_neighbour_counts[residue_id]:
            max_neighbour_counts[residue_id] = neighbour_count

    # Filter the original data to retain only one entry for each 'residue_id' with the highest 'neighbour_count'
    filtered_data = []
    for atom in data:
        residue_id = atom['residue_id']
        neighbour_count = atom.get('neighbour_count', 0)
        if neighbour_count == max_neighbour_counts[residue_id]:
            # Remove other entries for the same residue_id
            filtered_data = [entry for entry in filtered_data if entry['residue_id'] != residue_id]
            filtered_data.append(atom)
            atom.pop('model_id')
            atom.pop('atom_name')
            atom.pop('atom_coords')

    return filtered_data

def amino_acid_vector(amino_acid_info_count):
    result = []
    for amino_acid in amino_acid_info_count:
        entry = [f"{amino_acid['residue_name']}_{amino_acid['residue_id']}", amino_acid['neighbour_count']]
        result.append(entry)
    return result

In [106]:
def find_nearest_neighbors(data):
    """
    Find the n-neighbors closest to the query_point in the data using brute force with Euclidean distance.

    Parameters:
    - data: np.ndarray, dataset with shape (num_points, num_features)
    - query_point: np.ndarray, the point to find neighbors for with shape (num_features,)
    - n_neighbors: int, number of neighbors to find

    Returns:
    - indices: np.ndarray, indices of the nearest neighbors in the dataset
    - distances: np.ndarray, distances of the nearest neighbors from the query_point
    """
    start_time = time.time()

    coords = np.array([atom['atom_coords'] for atom in data])
    neighbour_counts = np.zeros(len(data), dtype=int)

    # Calculate Euclidean distances from the query point to all other points
    index = 0
    for query_point in coords:
        neighbour_count = -1

        distances = scipy.spatial.distance.cdist([query_point], coords, 'euclidean').flatten()
        for distance in distances:
            if distance <= 3:
                neighbour_count += 1;
        
        neighbour_counts[index] = neighbour_count
        data[index]['neighbour_count'] = neighbour_count
        index += 1

    end_time = time.time()
    elapsed = end_time - start_time
    print(elapsed)

    return data

In [107]:
pdb_file_path = './pdb/pdb3epz.pdb'
amino_vector = []

atom_info_list = extract_pdb_info(pdb_file_path)
print(len(atom_info_list))
# atom_info_count = count_neighboring_atoms(atom_info_list[:])
atom_info_count = find_nearest_neighbors(atom_info_list[:])
print(atom_info_count)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8012.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 8032.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8033.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 8100.
  warnings.warn(


3550
0.8123219013214111
[{'model_id': 0, 'chain_id': 'A', 'residue_name': 'GLY', 'residue_id': 350, 'atom_name': 'N', 'atom_coords': [33.525001525878906, 15.960000038146973, 117.6510009765625], 'neighbour_count': 3}, {'model_id': 0, 'chain_id': 'A', 'residue_name': 'GLY', 'residue_id': 350, 'atom_name': 'CA', 'atom_coords': [33.64500045776367, 17.110000610351562, 116.70700073242188], 'neighbour_count': 5}, {'model_id': 0, 'chain_id': 'A', 'residue_name': 'GLY', 'residue_id': 350, 'atom_name': 'C', 'atom_coords': [32.83300018310547, 16.937999725341797, 115.43000030517578], 'neighbour_count': 6}, {'model_id': 0, 'chain_id': 'A', 'residue_name': 'GLY', 'residue_id': 350, 'atom_name': 'O', 'atom_coords': [32.27399826049805, 15.859999656677246, 115.18599700927734], 'neighbour_count': 5}, {'model_id': 0, 'chain_id': 'A', 'residue_name': 'PRO', 'residue_id': 351, 'atom_name': 'N', 'atom_coords': [32.763999938964844, 18.000999450683594, 114.59700012207031], 'neighbour_count': 9}, {'model_id': 

In [110]:
def pdb_to_n_neighbour_vector(pdb_file_path):
    '''Step 1: Counting n-neighbours for atoms'''
        # convert_cif_to_pdb(cif_file_path, pdb_file_path)
    amino_vector = []

    atom_info_list = extract_pdb_info(pdb_file_path)
    print(len(atom_info_list))

    num_atom_entries = 100 #Adjust this, total is 7714 atoms
    # atom_info_count = count_neighboring_atoms(atom_info_list[:])
    atom_info_count = find_nearest_neighbors(atom_info_list[:])

    '''
    #Check loading
    for atom_info in atom_info_count[:5]:
        print(atom_info)
    '''

    '''Step 2: Counting n-neighbours for amino acids'''
    amino_acid_info_count = count_neighbouring_amino_acids(atom_info_count)

    '''
    #Check amino_acid_neighbour, should have 1389
    num_amino_acid_entries = 20 #Adjust this
    for amino_acid in amino_acid_info_count[:num_amino_acid_entries]:  
        print(f"{amino_acid}")
    '''

    print(f"Number of Amino Acids: {len(amino_acid_info_count)}")

    '''Step 3: Return [amino acid, n-neighbours] 2D vector'''
    indiv_amino_vector = amino_acid_vector(amino_acid_info_count)
    print(indiv_amino_vector)

    for amino_acid in indiv_amino_vector:
        amino_vector.append(amino_acid)

    with open('20-amino-vector-fast', 'a') as f:
        f.write(str(amino_vector))

    result_queue.put(amino_vector)

    return(amino_vector)

In [109]:
def merge_amino_vector(residues):
    # Use a dictionary to store the highest value for each residue
    residue_dict = {}
    
    for residue in residues:
        name, count = residue
        if name in residue_dict:
            if count > residue_dict[name]:
                residue_dict[name] = count
        else:
            residue_dict[name] = count

    # Convert the dictionary back to a list of lists
    result = [[name, count] for name, count in residue_dict.items()]
    
    # Sort the list by residue_id in ascending order
    result.sort(key=lambda x: int(x[0].split('_')[1]))
    
    return result

def plot_residues(df):
    plt.figure(figsize=(15, 8))  # Set the figure size
    plt.bar(df['Residue'], df['Neighbour_Count'], color='skyblue')
    plt.xlabel('Residue')
    plt.ylabel('Neighbour Count')
    plt.title('Residue Neighbour Count')
    plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
    plt.tight_layout()  # Adjust layout to fit x-axis labels
    plt.show()

In [111]:
cif_file_path = './7sfc_updated.cif'
pdb_file_path = ['./pdb/pdb3epz.pdb', './pdb/pdb3pta.pdb', './pdb/pdb3swr.pdb', './pdb/pdb4wxx.pdb',
                 './pdb/pdb4yoc.pdb', './pdb/pdb4z96.pdb', './pdb/pdb4z97.pdb', './pdb/pdb5wvo.pdb',
                 './pdb/pdb5ydr.pdb', './pdb/pdb6k3a.pdb', './pdb/pdb6l1f.pdb', './pdb/pdb6x9i.pdb',
                 './pdb/pdb6x9j.pdb', './pdb/pdb6x9k.pdb', './pdb/pdb7sfc.pdb', './pdb/pdb7sfe.pdb',
                 './pdb/pdb7sff.pdb', './pdb/pdb7sfg.pdb', './pdb/pdb8v9u.pdb', './pdb/AF-P26358-F1-model_v4.pdb']

# ============== MAIN =======================

#pdb_file_path1 = ['./pdb/AF-P26358-F1-model_v4.pdb']
#pdb_file_path2 = ['./pdb/pdb7sfe.pdb']
#pdb_file_path3 = ['./pdb/pdb7sff.pdb']
#pdb_file_path4 = ['./pdb/pdb7sfg.pdb']
#pdb_file_path5 = ['./pdb/pdb8v9u.pdb']
#pdb_file_path6 = ['./pdb/pdb8v9u.pdb']
#pdb_file_path7 = ['./pdb/pdb8v9u.pdb']
#pdb_file_path8 = ['./pdb/pdb6x9k.pdb']


import threading
from queue import Queue
import os

start_time = time.time()
result_queue = Queue()

thread1 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[0],))
thread2 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[1],))
thread3 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[2],))
thread4 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[3],))
thread5 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[4],))
thread6 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[5],))
thread7 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[6],))
thread8 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[7],))
thread9 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[8],))
thread10 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[9],))
thread11 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[10],))
thread12 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[11],))
thread13 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[12],))
thread14 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[13],))
thread15 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[14],))
thread16 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[15],))
thread17 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[16],))
thread18 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[17],))
thread19 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[18],))
thread20 = threading.Thread(target=pdb_to_n_neighbour_vector, args=(pdb_file_path[19],))

thread1.start()
thread2.start()
thread3.start()
thread4.start()
thread5.start()
thread6.start()
thread7.start()
thread8.start()
thread9.start()
thread10.start()
thread11.start()
thread12.start()
thread13.start()
thread14.start()
thread15.start()
thread16.start()
thread17.start()
thread18.start()
thread19.start()
thread20.start()

thread1.join()
thread2.join()
thread3.join()
thread4.join()
thread5.join()
thread6.join()
thread7.join()
thread8.join()
thread9.join()
thread10.join()
thread11.join()
thread12.join()
thread13.join()
thread14.join()
thread15.join()
thread16.join()
thread17.join()
thread18.join()
thread19.join()
thread20.join()

result = result_queue.get()
print(type(result))
print(result)

#with open('20-amino-vector-fast', 'w') as f:
    #f.write(str(result))

end_time = time.time()
elapsed = end_time - start_time
print(elapsed)

#print(thread1)
#print(thread2)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8012.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 8032.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8033.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 8100.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 3684.
  warnings.w

3550
667
3334
3339
4097


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8815.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 8646.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 6322.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 6328.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 6330.
  warnings.w

79894094
5671

7907
7383
11381
7461


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 14886.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 14952.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 14956.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 14988.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 15369.
  warni

7130
7714
6956
7255
7277
12862


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 14796.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 14866.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 14904.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 14934.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 15120.
  warni

19035
40590
5.409203052520752
Number of Amino Acids: 105
[['ARG_140', 6], ['SER_141', 8], ['MLZ_142', 8], ['SER_143', 9], ['ASP_144', 9], ['GLY_145', 5], ['HIS_3', 6], ['MET_4', 9], ['PRO_5', 10], ['PRO_6', 9], ['ASN_7', 9], ['ARG_8', 9], ['PRO_9', 9], ['GLY_10', 6], ['ILE_11', 9], ['THR_12', 9], ['PHE_13', 8], ['GLU_14', 8], ['ILE_15', 9], ['GLY_16', 7], ['ALA_17', 8], ['ARG_18', 8], ['LEU_19', 8], ['GLU_20', 9], ['ALA_21', 8], ['LEU_22', 8], ['ASP_23', 9], ['TYR_24', 9], ['LEU_25', 9], ['GLN_26', 10], ['LYS_27', 9], ['TRP_28', 9], ['TYR_29', 9], ['PRO_30', 9], ['SER_31', 9], ['ARG_32', 8], ['ILE_33', 9], ['GLU_34', 8], ['LYS_35', 8], ['ILE_36', 9], ['ASP_37', 9], ['TYR_38', 8], ['GLU_39', 8], ['GLU_40', 10], ['GLY_41', 8], ['LYS_42', 9], ['MET_43', 9], ['LEU_44', 8], ['VAL_45', 9], ['HIS_46', 8], ['PHE_47', 9], ['GLU_48', 8], ['ARG_49', 8], ['TRP_50', 9], ['SER_51', 9], ['HIS_52', 9], ['ARG_53', 9], ['TYR_54', 10], ['ASP_55', 9], ['GLU_56', 8], ['TRP_57', 9], ['ILE_58', 9], ['TYR_59'

In [ ]:
# split amino vector
with open('amino_vector', 'r') as f:
    amino_vector_string = f.read()
    elements = re.findall(r'\[.*?\]|\w+', amino_vector_string)
    output_list = [eval(element) if element.startswith('[') else element for element in elements]

    print(len(output_list))
    print(output_list)

SyntaxError: '[' was never closed (<string>, line 1)

In [ ]:
import re

# Merge 20-amino-vector-unmerged
with open('20-amino-vector-unmerged-copy', 'r') as f:
    amino_vector = f.read()
    pattern = r"\['(\w+_\d+)', (\d+)\]"
    matches = re.findall(pattern, amino_vector)
    amino_vector_list = [[match[0], int(match[1])] for match in matches]
    merged_amino_vector = merge_amino_vector(amino_vector_list)

    print(merged_amino_vector)
    #with open('20-amino-vector', 'a') as g:
        #g.write(str(merged_amino_vector))

    amino_codes = ['ALA', 'ARG', 'ASN', 'ASP', 'ASX', 'CYS', 'GLN', 'GLU', 'GLX', 
            'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 
            'THR', 'TRP', 'TYR', 'UNK', 'VAL'] 

    acid_only_data = [entry for entry in merged_amino_vector if entry[0].split('_')[0] in amino_codes]

print(len(list))

amino_df = pd.DataFrame(acid_only_data, columns=['Residue', 'Neighbour_Count'])
filtered_df = amino_df[amino_df['Neighbour_Count'] >= 0]

print(filtered_df)
print(filtered_df.shape)

#print(amino_df)
#print(amino_df.shape)

amino_df.to_csv('DNMT1-acid-only.csv', index=False)

# plot_residues(amino_df)


[['HIS_0', 5], ['MET_1', 6], ['NA_1', 0], ['DG_1', 7], ['SO4_1', 4], ['DT_1', 6], ['ARG_2', 5], ['GLN_2', 5], ['HOH_2', 0], ['PHE_2', 5], ['DA_2', 7], ['MES_2', 5], ['DC_2', 7], ['PRO_2', 6], ['HIS_3', 5], ['THR_3', 6], ['ILE_3', 6], ['HOH_3', 2], ['GLU_3', 5], ['DG_3', 7], ['ZN_3', 0], ['DC_3', 6], ['ALA_3', 4], ['MET_4', 6], ['PHE_4', 6], ['HOH_4', 0], ['ALA_4', 5], ['DG_4', 7], ['MES_4', 5], ['DC_4', 6], ['ARG_4', 5], ['PRO_5', 7], ['VAL_5', 6], ['ARG_5', 5], ['DC_5', 6], ['HOH_5', 1], ['DG_5', 7], ['THR_5', 6], ['PRO_6', 6], ['THR_6', 6], ['LYS_6', 6], ['HOH_6', 1], ['LEU_6', 5], ['5CM_6', 9], ['DT_6', 6], ['ALA_6', 5], ['ASN_7', 6], ['THR_7', 7], ['HOH_7', 0], ['VAL_7', 6], ['DG_7', 7], ['PRO_7', 6], ['ARG_8', 6], ['LEU_8', 6], ['GLN_8', 6], ['DC_8', 7], ['HOH_8', 2], ['DA_8', 6], ['ALA_8', 4], ['PRO_9', 6], ['THR_9', 6], ['HOH_9', 1], ['GLY_9', 4], ['DC_9', 6], ['DG_9', 7], ['ARG_9', 5], ['GLY_10', 4], ['SER_10', 6], ['HOH_10', 1], ['DT_10', 6], ['DC_10', 6], ['VAL_10', 6], ['ILE

In [ ]:
# Function to display most connected atoms onto PDB view
sum([1,16,190,1153,961,157,126,115,376,556])

3651

amino_codes = ['ALA', 'ARG', 'ASN', 'ASP', 'ASX', 'CYS', 'GLN', 'GLU', 'GLX', 
        'GLY', 'HIS', 'ILE', 'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 
        'THR', 'TRP', 'TYR', 'UNK', 'VAL'] 

acid_only_data = [entry for entry in data if entry[0].split('_')[0] in amino_codes]

print(len(list))